# ENVIRONMENT

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain youtube-transcript-api pytube

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
if os.getenv("OPENAI_API_KEY"):
    os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

## MULTI-REPRESENTATION INDEX

Instead of embedding raw chunks — embed summaries. Retrieve full documents using those summaries.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

Here we load two documents and add both of them in docs.

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="openrouter/owl-alpha",max_retries=0)
    | StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})

Here we are making the chain and making connection. And variable summaries is a list of all summaries of document here it is two.

In [ ]:
from langchain_core.stores import InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import MultiVectorRetriever

vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())

store = InMemoryByteStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

doc_ids = [str(uuid.uuid4()) for _ in docs]

summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In this block we first embed the summaries and store it in chroma then variable named store that store documents in memory then we define the unique id to each document summary that can be link to document then we link the document with that unique id and after that we can retrieve the information.

In [ ]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query,k=1)
sub_docs[0]

In [ ]:
retrieved_docs = retriever.invoke(query)
retrieved_docs[0].page_content[0:500]

### How Multi-Representation Indexing Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     MULTI-REPRESENTATION INDEXING                       │
└─────────────────────────────────────────────────────────────────────────┘

  ┌──────────────┐         LLM Summarization         ┌──────────────────┐
  │  Full Docs   │ ──────────────────────────────────►│   Summaries      │
  │  (Doc 1..N)  │                                    │   (Doc 1..N)     │
  └──────┬───────┘                                    └────────┬─────────┘
         │                                                     │
         │  Assign shared doc_id                               │  Embed summaries
         │  to each pair                                       │  + attach doc_id
         ▼                                                     ▼
  ┌──────────────────┐                              ┌─────────────────────┐
  │   RAM Store      │                              │   Chroma VectorDB   │
  │  (InMemoryByte   │                              │  (Summary Vectors)  │
  │   Store)         │                              │                     │
  │                  │         Linked by            │                     │
  │  doc_id ──► Full │◄─────── doc_id ─────────────►│  doc_id ──► Summary │
  │        Document  │                              │         Embedding   │
  └──────────────────┘                              └─────────────────────┘
         ▲                                                     │
         │                                                     │
         │              ┌───────────────────────┐              │
         │              │     User Query        │              │
         │              │  "Memory in agents"   │              │
         │              └───────────┬───────────┘              │
         │                          │                          │
         │                          ▼                          │
         │              ┌───────────────────────┐              │
         │              │  1. Similarity Search  │◄────────────┘
         │              │     in Chroma          │
         │              │  (search summaries)    │
         │              └───────────┬───────────┘
         │                          │
         │                          │  Returns matched doc_id
         │                          ▼
         │              ┌───────────────────────┐
         └──────────────│  2. Retrieve Full Doc  │
                        │     from RAM Store     │
                        │  (using doc_id)        │
                        └───────────┬───────────┘
                                    │
                                    ▼
                        ┌───────────────────────┐
                        │  3. Return Full        │
                        │     Document to User   │
                        └───────────────────────┘
```

**Key Insight:**
- **Chroma** holds summary vectors → used for *searching* (semantic similarity)
- **RAM Store** holds full documents → used for *returning* (complete content)
- Both are linked by **`doc_id`** — search happens in Chroma, but the full document is returned from the RAM store

## RAPTOR

Clusters and summarizes chunks at multiple levels. Builds a tree of summaries for better retrieval.

### How RAPTOR Indexing & Retrieval Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                             RAPTOR INDEXING                             │
└─────────────────────────────────────────────────────────────────────────┘

  [Level 2 Summaries]               ┌────────────────────────┐
                                    │  Summary of Summaries  │
                                    └───────────▲────────────┘
                                                │
                                                │ Summarize clusters
                                                │
  [Level 1 Summaries]               ┌───────────┴────────────┐
                                    │ Summary A  │ Summary B │
                                    └─────▲────────────▲─────┘
                                          │            │
                                          │ Group &    │ Summarize
                                          │ Summarize  │
                                          │            │
  [Level 0 Raw Chunks]          ┌─────────┴───┐    ┌───┴─────────┐
                                │ Chunk 1 & 2 │    │   Chunk 3   │
                                └─────────────┘    └─────────────┘

───────────────────────────────────────────────────────────────────────────
                            RAPTOR RETRIEVAL
───────────────────────────────────────────────────────────────────────────

                                ┌───────────────────┐
                                │    User Query     │
                                └─────────┬─────────┘
                                          │
                                          ▼
                                ┌───────────────────┐
                                │ Search across ALL │◄── Query compared to
                                │ tree levels       │    all levels at once
                                └─────────┬─────────┘
                                          │
                        ┌─────────────────┴─────────────────┐
                        │                                   │
                        ▼                                   ▼
              ┌───────────────────┐               ┌───────────────────┐
              │ Specific Question │               │  Broad Question   │
              │  (e.g., details)  │               │  (e.g., overview) │
              └─────────┬─────────┘               └─────────┬─────────┘
                        │                                   │
                        ▼                                   ▼
              ┌───────────────────┐               ┌───────────────────┐
              │ Matches Low-Level │               │Matches High-Level │
              │ Chunk (Level 0)   │               │Summary (Level 2/1)│
              └───────────────────┘               └───────────────────┘
```

**Key Insights:**
- **Multi-Level Tree of Summaries**: Builds a tree of summaries where **Level 0** is the raw chunks, **Level 1** consists of similar chunks grouped and summarized, and **Level 2** consists of similar summaries grouped and summarized again.
- **Collapsing / Flat Search**: Query searches across all nodes (chunks and summaries) of the tree simultaneously.
- **Specific vs. Broad Matches**: Specific questions match low-level details (Level 0 chunks) while broad questions match high-level summaries (Level 1/2).

## COLBERT

Embeds each token of a document separately for fine-grained, token-level matching between queries and documents.

- **Standard** → one vector per chunk → average matching
- **ColBERT**  → one vector per token → precise matching

### How ColBERT Retrieval Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          ColBERT RETRIEVAL                              │
└─────────────────────────────────────────────────────────────────────────┘

  ┌──────────────────┐
  │  Document Loaded  │
  └────────┬─────────┘
           │
           │  Tokenize & embed each token individually
           ▼
  ┌──────────────────────────────────────────────────┐
  │          Document Token Embeddings                │
  │  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐       │
  │  │ T1  │ │ T2  │ │ T3  │ │ T4  │ │ ... │       │
  │  │ vec │ │ vec │ │ vec │ │ vec │ │ vec │       │
  │  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘       │
  └──────────────────────────────────────────────────┘

                                ┌───────────────────┐
                                │    User Query      │
                                └─────────┬─────────┘
                                          │
                                          │  Tokenize & embed each token
                                          ▼
                                ┌──────────────────────────┐
                                │  Query Token Embeddings   │
                                │  ┌─────┐ ┌─────┐ ┌─────┐│
                                │  │ Q1  │ │ Q2  │ │ Q3  ││
                                │  │ vec │ │ vec │ │ vec ││
                                │  └─────┘ └─────┘ └─────┘│
                                └────────────┬─────────────┘
                                             │
                                             ▼
  ┌──────────────────────────────────────────────────────────────────┐
  │              MaxSim (Maximum Similarity) Scoring                  │
  │                                                                   │
  │   For each query token Qi:                                        │
  │     Find the BEST matching document token Tj                      │
  │     Score_i = max( sim(Qi, T1), sim(Qi, T2), ... , sim(Qi, Tn) ) │
  │                                                                   │
  │   Total Score = Score_1 + Score_2 + Score_3                        │
  └──────────────────────────────┬───────────────────────────────────┘
                                 │
                                 ▼
                    ┌───────────────────────┐
                    │  Rank Chunks by Total  │
                    │  Score (highest first) │
                    └───────────┬───────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │  Return Most Relevant  │
                    │  Chunks to User        │
                    └───────────────────────┘
```

**Key Insights:**
- **Token-Level Embeddings**: Unlike standard retrieval (one vector per chunk), ColBERT creates one vector per token for both documents and queries.
- **MaxSim Scoring**: Each query token finds its best matching document token — scores are then summed for a total relevance score.
- **Precision**: This allows ColBERT to capture fine-grained semantic matches that single-vector approaches may miss.